In [2]:

"""
Zero-Shot and Three-Shot ATT&CK Classification
"""

import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter

#  Paths 
RESULTS_DIR   = Path("../results")
COMMUNITY_CSV = RESULTS_DIR / "community_assignments.csv"
STIX_FILE     = Path("../data/attck/enterprise-attack.json")
OUTPUT_DIR    = RESULTS_DIR / "stage4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#  Parameters 
OLLAMA_URL      = "http://localhost:11434/api/generate"
MODELS          = ["qwen2.5:3b"]
CONDITIONS      = ["zero_shot", "three_shot"]
RANDOM_SEED     = 1234
TEMPERATURE     = 0.0
MAX_COMMUNITY   = 50
MIN_COMM_SIZE   = 5
TIMEOUT         = 120

# ATT&CK candidates relevant to CIC-IDS 2017
CANDIDATES = """
CANDIDATE TECHNIQUES (select the most appropriate one):
- T1046     | Network Service Discovery  | Discovery          | Port scanning and service enumeration
- T1110.001 | Password Guessing          | Credential Access  | Brute force login attempts
- T1498.001 | Direct Network Flood       | Impact             | High-volume DDoS flooding
- T1499.001 | OS Exhaustion Flood        | Impact             | DoS via resource exhaustion
- T1071.001 | Web Protocols              | Command And Control| C2 communication over HTTP/HTTPS
- T1105     | Ingress Tool Transfer      | Command And Control| Downloading tools or payloads
- T1059.007 | JavaScript                 | Execution          | Malicious script execution via browser
- T1190     | Exploit Public-Facing App  | Initial Access     | Web application exploitation
"""

VALID_IDS = [
    "T1046", "T1110.001", "T1498.001", "T1499.001",
    "T1071.001", "T1105", "T1059.007", "T1190"
]

# Load community assignments 
def load_communities():
    print("Loading community assignments...")
    df = pd.read_csv(COMMUNITY_CSV, low_memory=False)
    print(f"  Total alerts     : {len(df):,}")
    print(f"  Communities      : {df['community_id'].nunique()}")

    # Filter to meaningful communities
    comm_sizes  = df["community_id"].value_counts()
    valid_comms = comm_sizes[comm_sizes >= MIN_COMM_SIZE].index.tolist()
    df          = df[df["community_id"].isin(valid_comms)].copy()

    # Remove benign communities — nothing to classify
    df = df[df["community_dominant_tactic"] != "Benign"].copy()
    print(f"  Attack communities (size>={MIN_COMM_SIZE}): "
          f"{df['community_id'].nunique()}")

    # Stratified sample up to MAX_COMMUNITY
    np.random.seed(RANDOM_SEED)
    sampled_comms = []
    tactics       = df["community_dominant_tactic"].unique()
    per_tactic    = max(1, MAX_COMMUNITY // len(tactics))

    for tactic in tactics:
        tactic_comms = df[
            df["community_dominant_tactic"] == tactic
        ]["community_id"].unique().tolist()
        n       = min(per_tactic, len(tactic_comms))
        sampled = np.random.choice(
            tactic_comms, size=n, replace=False
        ).tolist()
        sampled_comms.extend(sampled)

    sampled_comms = sampled_comms[:MAX_COMMUNITY]
    df            = df[df["community_id"].isin(sampled_comms)].copy()
    print(f"  Communities to classify : {len(sampled_comms)}")
    print(f"  Alerts in those         : {len(df):,}")
    return df, sampled_comms

# Load ATT&CK STIX for verification 
def load_attck_techniques():
    print("\nLoading ATT&CK STIX bundle for verification...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)
    techniques = {}
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        if obj.get("revoked", False) or obj.get("deprecated", False):
            continue
        tech_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id")
                break
        if not tech_id:
            continue
        tactics = []
        for phase in obj.get("kill_chain_phases", []):
            if phase.get("kill_chain_name") == "mitre-attack":
                tactics.append(
                    phase["phase_name"].replace("-", " ").title()
                )
        techniques[tech_id] = {
            "name":   obj.get("name", ""),
            "tactic": tactics[0] if tactics else "Unknown",
        }
    print(f"  Loaded {len(techniques)} techniques from STIX")
    return techniques

# Step 3: Build community description 
def build_community_description(comm_df):
    """
    Build a structured natural language description of a community.
    Includes dominant IDS label, flow statistics, TCP flags,
    and automated behavioural indicators.
    """
    n = len(comm_df)

    def safe_mean(col):
        if col not in comm_df.columns:
            return 0.0
        vals = pd.to_numeric(comm_df[col], errors="coerce").dropna()
        return round(float(vals.mean()), 1) if len(vals) > 0 else 0.0

    # Dominant IDS label — strongest signal
    label_counts   = comm_df["Label"].value_counts()
    dominant_label = label_counts.index[0]
    label_pct      = label_counts.iloc[0] / n * 100
    label_summary  = (
        f"Dominant IDS label: {dominant_label} "
        f"({label_counts.iloc[0]} of {n} alerts, {label_pct:.0f}%)"
    )

    # Port distribution
    if "Destination Port" in comm_df.columns:
        top_ports = comm_df["Destination Port"].value_counts().head(3)
        port_str  = ", ".join(
            [f"port {p} ({c} flows)" for p, c in top_ports.items()]
        )
    else:
        port_str = "unknown"

    # Flow statistics
    duration = safe_mean("Flow Duration")
    fwd_pkts = safe_mean("Total Fwd Packets")
    bwd_pkts = safe_mean("Total Backward Packets")
    pkt_mean = safe_mean("Packet Length Mean")
    pkt_std  = safe_mean("Packet Length Std")
    flow_bps = safe_mean("Flow Bytes/s")
    syn      = safe_mean("SYN Flag Count")
    fin      = safe_mean("FIN Flag Count")
    rst      = safe_mean("RST Flag Count")
    psh      = safe_mean("PSH Flag Count")
    ack      = safe_mean("ACK Flag Count")
    init_fwd = safe_mean("Init_Win_bytes_forward")
    init_bwd = safe_mean("Init_Win_bytes_backward")

    # TCP flags string
    flags = []
    if syn  > 0.1: flags.append(f"SYN(avg={syn:.1f})")
    if fin  > 0.1: flags.append(f"FIN(avg={fin:.1f})")
    if rst  > 0.1: flags.append(f"RST(avg={rst:.1f})")
    if psh  > 0.1: flags.append(f"PSH(avg={psh:.1f})")
    if ack  > 0.1: flags.append(f"ACK(avg={ack:.1f})")
    flag_str = ", ".join(flags) if flags else "none prominent"

    # Day distribution
    if "day" in comm_df.columns:
        days    = comm_df["day"].value_counts().to_dict()
        day_str = ", ".join([f"{d}({c})" for d, c in days.items()])
    else:
        day_str = "unknown"

    # Behavioural indicators
    clues = []
    if fwd_pkts > 1 and bwd_pkts < 0.5:
        clues.append(
            "One-directional traffic (high forward, near-zero backward) "
            "suggests a flood or scan with no server response."
        )
    if flow_bps > 500_000:
        clues.append(
            f"High flow rate ({flow_bps/1_000_000:.1f} Mbps) "
            "is consistent with volumetric denial-of-service."
        )
    if syn > 0.5 and fin < 0.1 and ack < 0.1:
        clues.append(
            "SYN flags with no FIN/ACK completions indicates "
            "incomplete TCP handshakes, consistent with SYN flooding."
        )
    if rst > 0.5:
        clues.append(
            "High RST flag count suggests port scanning "
            "with connection rejections."
        )
    if duration < 100_000 and fwd_pkts < 3:
        clues.append(
            "Very short durations and low packet counts suggest "
            "automated scanning rather than legitimate sessions."
        )
    if "Destination Port" in comm_df.columns:
        top_port_pct = (
            comm_df["Destination Port"].value_counts().iloc[0] / n
        )
        if top_port_pct > 0.8:
            clues.append(
                f"{top_port_pct*100:.0f}% of flows target the same "
                "destination port, suggesting a targeted attack."
            )

    behaviour_str = (
        "\nBehavioural indicators:\n"
        + "\n".join(f"  * {c}" for c in clues)
        if clues else ""
    )

    # Sample alert texts
    texts      = comm_df["alert_text"].dropna().head(3).tolist()
    text_block = "\n".join([f"  - {t[:130]}" for t in texts])

    description = (
        f"Community size: {n} network flow alerts\n"
        f"{label_summary}\n"
        f"Destination ports: {port_str}\n"
        f"Mean flow duration: {duration:.0f} microseconds\n"
        f"Mean forward packets: {fwd_pkts}, "
        f"mean backward packets: {bwd_pkts}\n"
        f"Mean packet length: {pkt_mean} bytes (std {pkt_std})\n"
        f"Mean flow rate: {flow_bps:.0f} bytes/s\n"
        f"TCP flags observed: {flag_str}\n"
        f"Initial window bytes — forward: {init_fwd}, "
        f"backward: {init_bwd}\n"
        f"Days observed: {day_str}"
        f"{behaviour_str}\n"
        f"Sample alert texts:\n{text_block}"
    )
    return description

# Three-shot examples 
def build_three_shot_examples():
    """
    Three examples covering different ATT&CK tactics.
    """
    return """
EXAMPLE 1:
Community description: 158 alerts. Dominant IDS label: DDoS (158 of 158, 100%).
All flows target port 80. Very high flow rate. SYN flags only, no backward response.
Behavioural indicators: One-directional traffic suggests flood.
Output: {"technique_id": "T1498.001", "technique_name": "Direct Network Flood", "tactic": "Impact", "confidence": "high", "rationale": "High-volume one-directional SYN traffic to a single port with no server response is characteristic of a volumetric DDoS flood."}

EXAMPLE 2:
Community description: 210 alerts. Dominant IDS label: PortScan (210 of 210, 100%).
Sequential destination ports. RST responses. Very short durations, low packet counts.
Behavioural indicators: High RST count suggests port scanning with connection rejections.
Output: {"technique_id": "T1046", "technique_name": "Network Service Discovery", "tactic": "Discovery", "confidence": "high", "rationale": "Sequential port probing with RST responses and minimal payload is characteristic of automated network service enumeration."}

EXAMPLE 3:
Community description: 180 alerts. Dominant IDS label: FTP-Patator (180 of 180, 100%).
All flows target port 21 (FTP). Regular timing intervals. Multiple failed connections.
Behavioural indicators: Repeated connections to same port with no data transfer.
Output: {"technique_id": "T1110.001", "technique_name": "Password Guessing", "tactic": "Credential Access", "confidence": "high", "rationale": "Repeated automated FTP connection attempts with regular timing and no successful data transfer indicates brute force credential attack."}
"""

# Build prompts 
def build_zero_shot_prompt(description):
    return (
        "You are a cybersecurity analyst classifying network alert "
        "communities using MITRE ATT&CK.\n\n"
        f"COMMUNITY DESCRIPTION:\n{description}\n\n"
        f"{CANDIDATES}\n"
        "Select exactly ONE technique from the candidates above "
        "that best matches the community description.\n"
        "Return a JSON object with exactly these fields:\n"
        "- technique_id: the technique ID exactly as listed\n"
        "- technique_name: the technique name exactly as listed\n"
        "- tactic: the tactic exactly as listed\n"
        "- confidence: high, medium, or low\n"
        "- rationale: one sentence explaining your classification\n\n"
        "JSON only. No other text."
    )

def build_three_shot_prompt(description):
    examples = build_three_shot_examples()
    return (
        "You are a cybersecurity analyst classifying network alert "
        "communities using MITRE ATT&CK.\n\n"
        f"Here are three examples of correct classifications:\n{examples}\n"
        f"Now classify this community:\n\n"
        f"COMMUNITY DESCRIPTION:\n{description}\n\n"
        f"{CANDIDATES}\n"
        "Select exactly ONE technique from the candidates above "
        "that best matches the community description.\n"
        "Return a JSON object with exactly these fields:\n"
        "- technique_id: the technique ID exactly as listed\n"
        "- technique_name: the technique name exactly as listed\n"
        "- tactic: the tactic exactly as listed\n"
        "- confidence: high, medium, or low\n"
        "- rationale: one sentence explaining your classification\n\n"
        "JSON only. No other text."
    )

# Call Ollama 
def call_ollama(model, prompt):
    payload = {
        "model":  model,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "seed":        RANDOM_SEED,
            "num_predict": 350,
            "think":       False,
        }
    }
    try:
        resp = requests.post(
            OLLAMA_URL, json=payload, timeout=TIMEOUT
        )
        resp.raise_for_status()
        raw = resp.json().get("response", "").strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        if not raw:
            return None
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"    JSON error: {e}")
        return None
    except requests.exceptions.Timeout:
        print(f"    Timeout")
        return None
    except Exception as e:
        print(f"    API error: {e}")
        return None

#Evaluate prediction 
def evaluate_prediction(predicted, gt_technique, gt_tactic):
    if predicted is None:
        return {
            "exact_match": False,
            "fuzzy_match": False,
            "parse_failed": True,
            "predicted_id": None,
            "predicted_tactic": None,
            "predicted_name": None,
            "confidence": None,
            "rationale": None,
            "ground_truth_id": gt_technique,
            "ground_truth_tactic": gt_tactic,
            "valid_candidate": False,
        }

    pred_id     = str(predicted.get("technique_id", "")).strip().upper()
    pred_tactic = str(predicted.get("tactic", "")).strip().lower()
    gt_id       = str(gt_technique).strip().upper()
    gt_tac      = str(gt_tactic).strip().lower()

    # Normalise tactic strings for fuzzy comparison
    def norm(s):
        return s.replace(" ", "").replace("and", "").lower()

    exact = pred_id == gt_id
    fuzzy = norm(pred_tactic) == norm(gt_tac)

    return {
        "exact_match":        exact,
        "fuzzy_match":        fuzzy,
        "parse_failed":       False,
        "predicted_id":       pred_id,
        "predicted_tactic":   pred_tactic,
        "predicted_name":     predicted.get("technique_name", ""),
        "confidence":         predicted.get("confidence", ""),
        "rationale":          predicted.get("rationale", ""),
        "ground_truth_id":    gt_id,
        "ground_truth_tactic":gt_tac,
        "valid_candidate":    pred_id in [v.upper() for v in VALID_IDS],
    }

# Main classification loop 
def classify_communities(df, sampled_comms):
    all_results = []

    for model in MODELS:
        for condition in CONDITIONS:
            print(f"\n{'='*60}")
            print(f"  Model     : {model}")
            print(f"  Condition : {condition}")
            print(f"{'='*60}")

            results     = []
            exact_hits  = 0
            fuzzy_hits  = 0
            parse_fails = 0

            for comm_id in tqdm(
                sampled_comms, desc=f"  {model[:12]} {condition}"
            ):
                comm_df = df[df["community_id"] == comm_id].copy()
                if len(comm_df) < MIN_COMM_SIZE:
                    continue

                # Ground truth
                gt_tactic    = comm_df["community_dominant_tactic"].iloc[0]
                gt_technique = comm_df["attck_technique_id"].mode()[0]

                # Build description and prompt
                description = build_community_description(comm_df)
                if condition == "zero_shot":
                    prompt = build_zero_shot_prompt(description)
                else:
                    prompt = build_three_shot_prompt(description)

                # Call LLM
                start     = time.time()
                predicted = call_ollama(model, prompt)
                elapsed   = time.time() - start

                # Evaluate
                eval_r = evaluate_prediction(
                    predicted, gt_technique, gt_tactic
                )
                eval_r.update({
                    "community_id":  comm_id,
                    "community_size":len(comm_df),
                    "model":         model,
                    "condition":     condition,
                    "latency_s":     round(elapsed, 2),
                })
                results.append(eval_r)
                all_results.append(eval_r)

                if eval_r["parse_failed"]:
                    parse_fails += 1
                elif eval_r["exact_match"]:
                    exact_hits += 1
                elif eval_r["fuzzy_match"]:
                    fuzzy_hits += 1

            # Per-condition summary
            total = len(results)
            valid = total - parse_fails
            if valid > 0:
                print(f"\n  Results — {model} | {condition}:")
                print(f"    Communities classified : {total}")
                print(f"    Parse failures         : {parse_fails}")
                print(
                    f"    Exact match (technique): "
                    f"{exact_hits}/{valid} "
                    f"= {exact_hits/valid*100:.1f}%"
                )
                print(
                    f"    Fuzzy match (tactic)   : "
                    f"{exact_hits+fuzzy_hits}/{valid} "
                    f"= {(exact_hits+fuzzy_hits)/valid*100:.1f}%"
                )
                lats = [r["latency_s"] for r in results]
                print(f"    Mean latency           : {np.mean(lats):.1f}s")

            # Save per-condition CSV
            out_path = OUTPUT_DIR / (
                f"results_{model.replace(':','_')}"
                f"_{condition}.csv"
            )
            pd.DataFrame(results).to_csv(out_path, index=False)
            print(f"    Saved to: {out_path}")

    return all_results

# Summary 
def print_summary(all_results):
    df = pd.DataFrame(all_results)
    if df.empty:
        print("No results.")
        return

    print(f"\n")
    print(f"CLASSIFICATION SUMMARY")
    print(
        f"\n{'Model':<20} {'Condition':<12} "
        f"{'Exact%':>8} {'Fuzzy%':>8} "
        f"{'Valid%':>8} {'Fails':>6}"
    )
    print("-"*65)

    for (model, condition), group in df.groupby(["model", "condition"]):
        valid       = group[~group["parse_failed"]]
        total_valid = len(valid)
        if total_valid == 0:
            continue
        exact    = valid["exact_match"].sum()
        fuzzy    = valid["fuzzy_match"].sum()
        vcand    = valid["valid_candidate"].sum()
        fails    = group["parse_failed"].sum()
        tactic_acc = min((exact + fuzzy) / total_valid * 100, 100.0)
        print(
            f"  {model:<18} {condition:<12} "
            f"{exact/total_valid*100:>7.1f}% "
            f"{tactic_acc:>7.1f}% "
            f"{vcand/total_valid*100:>7.1f}% "
            f"{fails:>6}"
)
    # Save combined results
    combined_path = OUTPUT_DIR / "all_results.csv"
    df.to_csv(combined_path, index=False)
    print(f"\n  Full results saved to: {combined_path}")
    print(f"\n  Ready for Stage 5 — RAG-Augmented Incident Report Generation")

# Main 
def main():
    df, sampled_comms = load_communities()
    load_attck_techniques()
    all_results       = classify_communities(df, sampled_comms)
    print_summary(all_results)

if __name__ == "__main__":
    main()


Loading community assignments...
  Total alerts     : 10,673
  Communities      : 86
  Attack communities (size>=5): 18
  Communities to classify : 18
  Alerts in those         : 8,848

Loading ATT&CK STIX bundle for verification...
  Loaded 703 techniques from STIX

  Model     : qwen2.5:3b
  Condition : zero_shot


  qwen2.5:3b zero_shot: 100%|██████████| 18/18 [01:33<00:00,  5.21s/it]



  Results — qwen2.5:3b | zero_shot:
    Communities classified : 18
    Parse failures         : 0
    Exact match (technique): 13/18 = 72.2%
    Fuzzy match (tactic)   : 13/18 = 72.2%
    Mean latency           : 5.2s
    Saved to: ../results/stage4/results_qwen2.5_3b_zero_shot.csv

  Model     : qwen2.5:3b
  Condition : three_shot


  qwen2.5:3b three_shot: 100%|██████████| 18/18 [01:33<00:00,  5.19s/it]


  Results — qwen2.5:3b | three_shot:
    Communities classified : 18
    Parse failures         : 0
    Exact match (technique): 12/18 = 66.7%
    Fuzzy match (tactic)   : 12/18 = 66.7%
    Mean latency           : 5.2s
    Saved to: ../results/stage4/results_qwen2.5_3b_three_shot.csv


CLASSIFICATION SUMMARY

Model                Condition      Exact%   Fuzzy%   Valid%  Fails
-----------------------------------------------------------------
  qwen2.5:3b         three_shot      66.7%   100.0%   100.0%      0
  qwen2.5:3b         zero_shot       72.2%   100.0%   100.0%      0

  Full results saved to: ../results/stage4/all_results.csv

  Ready for Stage 5 — RAG-Augmented Incident Report Generation
